# Cleaning of speeches

After getting all the speeches of the laws we will work with, it's necessary to work on cleaning the speech for the analysis that come later.

### Imports and paths

In [2]:
# import chunk

# to install nltk: 
# %pip install nltk
import nltk
nltk.download('stopwords')

from pathlib import Path
import pandas as pd
import json
import re
from nltk.corpus import stopwords as nltk_stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jorge\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# set paths
LAWS_FOLDER = Path("../../data/extracted_laws")
SAVE_RESULTS = Path("../../data/processed/speeches.csv")

### Stopwords

In [4]:
# this stopwords come from nltk, it's a 10 year old list of common words but it's still useful.
spanish_stopwords = set(nltk_stopwords.words("spanish"))

# also we add some custom stopwords that are common in the speeches, this has been an iterative generated list of words.
custom_stopwords = {
    "señor", "señora", "señores", "señorita",
    "presidente", "presidenta",
    "diputado", "diputada", "diputados", "diputadas",
    "ministro", "ministra", "ministros", "ministras",
    "gracias", "he", "dicho", "qué", "proyecto", "ley", "despacha", "Presidente"
} # set are faster than lists for lookups

total_stopwords = spanish_stopwords.union(custom_stopwords)


### Loading the dicussions from the json files

In [5]:
def load_discussions(root):
    """
    Search recursively for all .json transcript files inside root, read each file and return a dataframe with metadata + transcript text.
    """
    root = Path(root)
    rows = []

    for json_path in root.rglob("*.json"): # rglob searches recursively for all the .json files withing the folder
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        

        # each json file has a law_meta and entry
        law_meta = data.get("law_meta", {})
        entry = data.get("entry", {})

        row = {
            # information about the file
            "file_name": json_path.name,
            "json_path": str(json_path),
            
            # information abouot the law itself
            "law_id": law_meta.get("law_id"), # this is how it is stored in the library of the congress
            "law_number": law_meta.get("law_number"), # this is the official number of the law
            "law_title": law_meta.get("law_title"),

            # information about the discussion
            "section_key": entry.get("section_key"),
            "date": entry.get("event_date"),

            # transcription of discussion
            "text": data.get("transcript_text"),
        }

        rows.append(row)

    return pd.DataFrame(rows)

# with this we load all the dicussions into a dataframe
discussions_df = load_discussions(LAWS_FOLDER)
print(len(discussions_df))

51


In [6]:
# discussions_df.info()
discussions_df.head(5)

,file_name,json_path,law_id,law_number,law_title,section_key,date,text
0,discussion_2022-06-07_1_5.json,..\..\data\extracted_laws\law_21472\discussion...,8035,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"Fecha 07 de junio, 2022. Diario de Sesión en S..."
1,discussion_2022-07-13_3_1.json,..\..\data\extracted_laws\law_21472\discussion...,8035,21.472,Crea un Fondo de Estabilización de Tarifas y e...,3-1,2022-07-13,"Fecha 13 de julio, 2022. Diario de Sesión en S..."
2,discussion_2023-07-10_2_9.json,..\..\data\extracted_laws\law_21678\discussion...,8309,21.678,Establece el acceso a internet como servicio p...,2-9,2023-07-10,"Fecha 10 de julio, 2023. Diario de Sesión en S..."
3,discussion_2025-03-03_2_2.json,..\..\data\extracted_laws\law_21733\discussion...,8373,21.733,"Modifica los cuerpos legales que indica, estab...",2-2,2025-03-03,"Discusión General. Fecha 03 de marzo, 2025. Of..."
4,discussion_2025-03-03_2_3.json,..\..\data\extracted_laws\law_21733\discussion...,8373,21.733,"Modifica los cuerpos legales que indica, estab...",2-3,2025-03-03,"Fecha 03 de marzo, 2025. Diario de Sesión en S..."


### Split the speeches
Each time someone speaks there is a clear pattern on how they appear. They are introduced and then they speak. That's why splitting into lines of speech is a good idea

In [7]:
def split_raw_turns(text):
    """
    Split one transcript into raw speaker turns, a new turn starts every time a speaker header appears.
    """

    lines = text.splitlines()
    turns = []

    current_turn = None
    turn_order = 0


    for line in lines:
        stripped = line.strip() # so we can match headers even if they have extra spaces

        # skip empty lines
        if stripped == "":
            continue

        header_match = re.compile(r"^(El|La)\s+(señor|señora|señorita)\s+(.+?)\.-\s*$").match(stripped)

        # if the line is a speaker header, start a new turn
        if header_match:
            # close the previous turn first
            if current_turn is not None:
                current_turn["speech_raw"] = current_turn["speech_raw"].strip()
                # we add the current turn to the list of turns
                turns.append(current_turn)

            turn_order += 1
            
            # extract the speaker's name and any parenthetical information
            speaker_body_raw = header_match.group(3)
            # in the parentheses we find information about the speaker, like role or first name
            parenthesis_match = re.search(r"\((.*?)\)", speaker_body_raw)
            parentheses = parenthesis_match.group(1).strip() if parenthesis_match else None


            current_turn = {
                "turn_order": turn_order,
                "speaker_header_raw": stripped,
                "speaker_article": header_match.group(1),   # El / La
                "speaker_body_raw": speaker_body_raw,  # the rest of the header
                "parentheses": parentheses,  # any parenthetical information
                "speech_raw": ""
            }

        # if it is not a header, add the line to the current speech
        else:
            if current_turn is not None:
                current_turn["speech_raw"] += stripped + "\n"

    # after the loop, save the last turn too
    if current_turn is not None:
        current_turn["speech_raw"] = current_turn["speech_raw"].strip()
        turns.append(current_turn)

    return turns

### Classification of speech

There are 3 clear categories of speech
* *floor control*
Comments made from roles like presidents or vicepresidents are always about organization of the chamber itself. It can also be recognised by the words said, like "Silence please", "The discussion is open", or related to elections, this are intended to control the floor, talk about rules, closing the speech of people that spoke too long, ask for silence, calling for elections

* *Guests*
To several sessions several political figures were invited or called to participate, mainly ministers or subsecretaries of Regional offices.
I consider the speech of the informant deputy in this space, this is the deputy that, representing a specific work group, brings a law to be discussed and voted on.

* *Substantive / political speech*
The main speech for this project. they have substantive weight because what theysay has a position and political goal, either to communicate and/or to convince people. In this case I will only consider deputies for Substantive speech, not guests



In [8]:
def classify_turn(parentheses, speech_raw):
    """
    Classification of turns
    """
    role = parentheses.lower() if parentheses is not None else ""
    speech = speech_raw.lower() if speech_raw is not None else ""

    # people controlling the floor
    if any(term in role for term in ["presidente", "Presidenta", "vicepresidente", "vicepresidenta", "Secretario", "secretario"]):
        return "floor_control"

    if any(term in speech for term in ["cerrado el debate", "en votación", "pido silencio"]):
        return "floor_control"

    # ministers
    if any(term in role for term in ["ministro", "ministra", "subsecretario", "subsecretaria"]):
        return "minister"

    if "informante" in speech or "de pie" in role:
        return "law_introduction"

    # everything else 
    return "substantive"

### Cleaning of speech
It will be limited to only substantive speeches

In [9]:
def clean_speech(text, turn_type):
    """
    Clean one speech text if its turn_type is substantive.
    """
    
    # skip rows that are not substantive
    if turn_type != "substantive":
        return None

    # lowercase
    text = text.lower()

    # replace line breaks and tabs with spaces
    text = text.replace("\n", " ").replace("\t", " ")

    # remove numbers
    text = re.sub(r"\d+", "", text)

    # normalize repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Remove punctuation while keeping letters and spaces
    text = re.sub(r"[^\w\sáéíóúñü]", " ", text)
    
    # tokenize very simply by spaces
    tokens = text.split()

    # remove stopwords and very short tokens
    tokens = [
        token for token in tokens
        if token not in total_stopwords and len(token) > 1
    ]

    # return cleaned text
    return " ".join(tokens)

In [10]:
def build_raw_turns_df(laws_df):
    """
    Apply split_raw_turns to every file in laws_df, and return one dataframe with all turns.
    """
    rows = []

    for _, row in laws_df.iterrows():
        
        # apply the split_raw_turns function
        turns = split_raw_turns(row["text"])

        for turn in turns:

            # apply classification and cleaning
            turn_type = classify_turn(turn["parentheses"], turn["speech_raw"])
            speech_clean = clean_speech(turn["speech_raw"], turn_type)

            rows.append({
                "law_number": row["law_number"],
                "law_title": row["law_title"],
                "section_key": row["section_key"],
                "date": row["date"],
                "speaker_header_raw": turn["speaker_header_raw"],
                "speaker_article": turn["speaker_article"],
                "speaker_body_raw": turn["speaker_body_raw"],
                "speech_raw": turn["speech_raw"],
                "speech_clean": speech_clean,
                "turn_type": turn_type
            })

    return pd.DataFrame(rows)

### Pipeline execution

In [11]:
# apply the function to build the dataframe with all the speeches
speeches_df = build_raw_turns_df(discussions_df)
speeches_df

,law_number,law_title,section_key,date,speaker_header_raw,speaker_article,speaker_body_raw,speech_raw,speech_clean,turn_type
0,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)","Iniciando el Orden del Día, corresponde tratar...",None,floor_control
1,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)","Doy la bienvenida al ministro de Energía, seño...",None,floor_control
2,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"La señorita AHUMADA, doña Yovana (de pie).-",La,"AHUMADA, doña Yovana (de pie)","Señor Presidente, en mi calidad de diputada in...",None,law_introduction
3,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,"El señor SOTO, don Raúl (Presidente).-",El,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado informante de la ...,None,floor_control
4,21.472,Crea un Fondo de Estabilización de Tarifas y e...,1-5,2022-06-07,El señor NARANJO (de pie).-,El,NARANJO (de pie),"Señor Presidente, honorable Cámara, en nombre ...",None,floor_control
...,...,...,...,...,...,...,...,...,...,...
2476,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Corresponde votar en particular el artículo 21...,None,floor_control
2477,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,None,floor_control
2478,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Corresponde votar en particular el artículo te...,None,floor_control
2479,21.800,Regula la extracción de áridos y modifica los ...,2-5,2024-12-17,El señor AEDO (Vicepresidente).-,El,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,None,floor_control


In [13]:
print(f"Total number of speeches is :{len(speeches_df)}\nTotal number of substantive speeches: {len(speeches_df[speeches_df["turn_type"] == "substantive"])}")

Total number of speeches is :2481
Total number of substantive speeches: 1111


Save the results

In [14]:
speeches_df.to_csv(SAVE_RESULTS, index=False, encoding="utf-8-sig")

### exploratory code

In [ ]:
# # check the context (lines surrounding desired text)
# def show_context(text, phrase, window=2):
#     """
#     Print the lines around a phrase
#     """
#     lines = text.splitlines() # split the text into lines

        # we loop through the lines and check if the phrase is in the line, if it is we print the lines around it
#     for i, line in enumerate(lines):
#         if phrase.lower() in line.lower():
#             start = max(0, i - window)
#             end = min(len(lines), i + window + 1)

#             print("\n")
#             for j in range(start, end):
#                 marker = ">>" if j == i else "  "
#                 print(f"{marker} {j}: {lines[j]}")

# sample_text = discussions_df.iloc[0]["text"]
# show_context(sample_text, "Tiene la palabra", window=2)

In [ ]:
# def collect_speaker_headers(discussions_df):
#     """
#     Find lines that look like speaker headers, return a dataframe with one row per header line.
#     """
#     rows = []

#     # we look for lines that start with "El señor", "La señora" or "La señorita", followed by a name and a dot-dash (".-")
#     for _, row in discussions_df.iterrows():
#         lines = row["text"].splitlines()

#         for i, line in enumerate(lines):
#             stripped = line.strip()

#             if re.compile(r"^(El|La)\s+(señor|señora|señorita)\s+(.+?)\.-\s*$").match(stripped):
#                 rows.append({
#                     "law_number": row["law_number"],
#                     "section_key": row["section_key"],
#                     "file_name": row["file_name"],
#                     "line_number": i,
#                     "speaker_header_raw": stripped
#                 })

#     return pd.DataFrame(rows)

# headers = collect_speaker_headers(discussions_df)
# headers

In [ ]:
# # check the percentage of starting phrases that have a parentheses
# total = len(headers)
# with_parentheses = headers["speaker_header_raw"].str.contains(r"[()]", regex=True).sum()
# without_parentheses = total - with_parentheses

# print(f"Total headers: {total}")
# print(f"With parentheses: {with_parentheses} ({with_parentheses/total*100:.1f}%)")
# print(f"Without parentheses: {without_parentheses} ({without_parentheses/total*100:.1f}%)")
